# 🏛️ TrueEngine V1: Zero-GPU High-Throughput Data Spawner (Kaggle CPU)
- 4 vCPUs fully saturated
- Stockfish tactical self-play & distillation
- Packaging into 100k chunks (.pt / .npz)
- Automatic upload to Hugging Face Dataset: `TrueHieu/trueengine-chess-dataset`
- Consumes **0 GPU Hours**


In [ ]:
# Setup dependencies & Stockfish
import subprocess
import sys
import os

print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-chess", "huggingface_hub", "tqdm"])
subprocess.run(["apt-get", "update", "-qq"])
subprocess.run(["apt-get", "install", "-y", "-qq", "stockfish"])

import chess
import chess.engine
import torch
import numpy as np
import time
import multiprocessing as mp
from huggingface_hub import HfApi

HF_TOKEN = os.getenv("HF_TOKEN") or None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN") or user_secrets.get_secret("HUGGINGFACE_TOKEN") or HF_TOKEN
except Exception:
    pass
if not HF_TOKEN:
    for token_path in [
        "/kaggle/working/token",
        os.path.expanduser("~/.cache/huggingface/token"),
        os.path.expanduser("~/.huggingface/token")
    ]:
        if os.path.exists(token_path):
            try:
                with open(token_path, "r") as f:
                    HF_TOKEN = f.read().strip()
                if HF_TOKEN:
                    break
            except Exception:
                pass
DATASET_REPO = "TrueHieu/trueengine-chess-dataset"

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=DATASET_REPO, repo_type="dataset", exist_ok=True)
print("Hugging Face Dataset ready:", DATASET_REPO)



In [ ]:
# Master Openings & Worker Self-Play Engine
MASTER_OPENINGS = [
    "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1", # 1. e4
    "rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR b KQkq d3 0 1", # 1. d4
    "rnbqkbnr/pppppppp/8/8/8/5N2/PPPPPPPP/RNBQKB1R b KQkq - 1 1", # 1. Nf3
    "rnbqkbnr/pppppppp/8/8/2P5/8/PP1PPPPP/RNBQKBNR b KQkq c3 0 1", # 1. c4
    # Sicilian Najdorf / Dragon
    "r1bqkbnr/pp1ppppp/2n5/2p5/4P3/5N2/PPPP1PPP/RNBQKB1R w KQkq - 2 3",
    # French Defense
    "rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2",
    # Caro-Kann
    "rnbqkbnr/pp1ppppp/2p5/8/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2",
    # Ruy Lopez
    "r1bqkbnr/pppp1ppp/2n5/1B2p3/4P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3",
    # King's Indian
    "rnbqkb1r/pppppp1p/5np1/8/2PP4/8/PP2PPPP/RNBQKBNR w KQkq - 0 3",
    # Queen's Gambit Declined
    "rnbqkb1r/ppp2ppp/4pn2/3p4/2PP4/2N5/PP2PPPP/R1BQKBNR w KQkq - 2 4",
    # Evans Gambit
    "r1bqk1nr/pppp1ppp/2n5/2b1p3/1PB1P3/5N2/P1PP1PPP/RNBQK2R b KQkq b3 0 4",
]

PIECE_TO_PLANE = {
    'P': 0, 'N': 1, 'B': 2, 'R': 3, 'Q': 4, 'K': 5,
    'p': 6, 'n': 7, 'b': 8, 'r': 9, 'q': 10, 'k': 11
}

def encode_fen(fen):
    board_part = fen.split()[0]
    tensor = np.zeros((12, 8, 8), dtype=np.float32)
    rows = board_part.split('/')
    for r_idx, row in enumerate(rows):
        c_idx = 0
        for char in row:
            if char.isdigit():
                c_idx += int(char)
            else:
                plane = PIECE_TO_PLANE.get(char)
                if plane is not None and c_idx < 8:
                    tensor[plane, r_idx, c_idx] = 1.0
                c_idx += 1
    return tensor



In [ ]:
# High-Speed Multi-Process Worker Function
import random

def worker_generate(worker_id, target_positions, queue):
    engine = chess.engine.SimpleEngine.popen_uci("/usr/games/stockfish")
    engine.configure({"Threads": 1, "Hash": 128})
    
    positions_collected = 0
    records = []
    
    while positions_collected < target_positions:
        fen = random.choice(MASTER_OPENINGS)
        board = chess.Board(fen)
        
        while not board.is_game_over() and len(board.move_stack) < 120 and positions_collected < target_positions:
            limit = chess.engine.Limit(depth=random.randint(12, 14), time=0.08)
            result = engine.play(board, limit, info=chess.engine.INFO_ALL)
            best_move = result.move
            if best_move is None:
                break
                
            info = engine.analyse(board, chess.engine.Limit(depth=12))
            score = info.get("score")
            wdl = 0.5
            if score:
                score_val = score.white().score(mate_score=10000)
                if score_val is not None:
                    wdl = 1.0 / (1.0 + 10 ** (-score_val / 400.0))
            
            encoded = encode_fen(board.fen())
            from_sq = best_move.from_square
            to_sq = best_move.to_square
            
            records.append({
                "state": encoded,
                "from_sq": from_sq,
                "to_sq": to_sq,
                "wdl": np.float32(wdl)
            })
            positions_collected += 1
            board.push(best_move)
            
            if len(records) >= 5000:
                queue.put(records)
                records = []
                
    if records:
        queue.put(records)
    engine.quit()
    queue.put(None) # Signal worker completion



In [ ]:
# Coordinator & Chunk Auto-Uploader
import os
import shutil

CHUNK_SIZE = 50000 # 50,000 positions per chunk (~20-25MB)
TOTAL_TARGET = 500000 # 500k target
NUM_WORKERS = 4
os.makedirs("/kaggle/working/chunks", exist_ok=True)

manager = mp.Manager()
queue = manager.Queue(maxsize=20)

target_per_worker = TOTAL_TARGET // NUM_WORKERS
processes = []
for wid in range(NUM_WORKERS):
    p = mp.Process(target=worker_generate, args=(wid, target_per_worker, queue))
    p.start()
    processes.append(p)

chunk_index = 1
current_states = []
current_froms = []
current_tos = []
current_wdls = []
completed_workers = 0
total_positions = 0
start_time = time.time()

print(f"Started {NUM_WORKERS} workers generating {TOTAL_TARGET} positions...")

while completed_workers < NUM_WORKERS:
    data = queue.get()
    if data is None:
        completed_workers += 1
        continue
        
    for item in data:
        current_states.append(item["state"])
        current_froms.append(item["from_sq"])
        current_tos.append(item["to_sq"])
        current_wdls.append(item["wdl"])
        total_positions += 1
        
        if len(current_states) >= CHUNK_SIZE:
            chunk_file = f"/kaggle/working/chunks/chunk_{chunk_index:03d}.npz"
            np.savez_compressed(
                chunk_file,
                states=np.array(current_states, dtype=np.float32),
                from_squares=np.array(current_froms, dtype=np.int16),
                to_squares=np.array(current_tos, dtype=np.int16),
                wdls=np.array(current_wdls, dtype=np.float32)
            )
            elapsed = time.time() - start_time
            pos_per_sec = total_positions / max(1, elapsed)
            print(f"[Chunk {chunk_index:03d}] Saved {len(current_states)} positions ({pos_per_sec:.1f} pos/s). Uploading to HF...", flush=True)
            
            try:
                api.upload_file(
                    path_or_fileobj=chunk_file,
                    path_in_repo=f"chunks/chunk_{chunk_index:03d}.npz",
                    repo_id=DATASET_REPO,
                    repo_type="dataset"
                )
                print(f"[Chunk {chunk_index:03d}] Uploaded successfully to Hugging Face!", flush=True)
                os.remove(chunk_file) # Clean disk space
            except Exception as e:
                print(f"[Warning] Failed uploading chunk: {e}")
                
            chunk_index += 1
            current_states = []
            current_froms = []
            current_tos = []
            current_wdls = []

# Final remaining positions
if current_states:
    chunk_file = f"/kaggle/working/chunks/chunk_{chunk_index:03d}.npz"
    np.savez_compressed(
        chunk_file,
        states=np.array(current_states, dtype=np.float32),
        from_squares=np.array(current_froms, dtype=np.int16),
        to_squares=np.array(current_tos, dtype=np.int16),
        wdls=np.array(current_wdls, dtype=np.float32)
    )
    api.upload_file(
        path_or_fileobj=chunk_file,
        path_in_repo=f"chunks/chunk_{chunk_index:03d}.npz",
        repo_id=DATASET_REPO,
        repo_type="dataset"
    )
    print(f"Final chunk {chunk_index:03d} uploaded!")

for p in processes:
    p.join()

print(f"Generation Complete! Total {total_positions} positions pushed to {DATASET_REPO}")

